# Step 1: Setup prerequisites

In [1]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

In [2]:
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI =  os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI)
# Check the connection to the server
mongodb_client.admin.command("ping")

{'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1773244672, 1),
  'signature': {'hash': b'\xd7\xe1\xca\xae\xc7\xf8\x05\xc0[\xd8+\xd9\x93?b\x8fl\xb96*',
   'keyId': 7610876245458288646}},
 'operationTime': Timestamp(1773244672, 1)}

In [ ]:
# Set the URL for our AI model proxy
PROXY_ENDPOINT = os.environ.get("PROXY_ENDPOINT")

In [ ]:
# Set the passkey provided by your workshop instructor
PASSKEY = "replace-with-passkey"

In [4]:
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env(["voyageai"], PASSKEY)

Successfully set VOYAGE_API_KEY environment variable.


# Step 2: Setup memory collections

In [5]:
import json
from utils import create_index, create_search_index, check_index_ready

### **Do not change the values assigned to the variables below**

In [ ]:
# Database name
DB_NAME = "mongodb_genai_devday_memory"
# Name of the chat history collection
CHATS_COLLECTION_NAME = "chats"
# Name of the semantic memory collection
SEMANTIC_COLLECTION_NAME = "semantic"
# Name of the episodic memory colleciton
EPISODIC_COLLECTION_NAME = "episodic"
# Name of the vector search index
VS_INDEX_NAME = "vector_index"

In [ ]:
# Access the `DB_NAME` database.
db = mongodb_client[DB_NAME]
# Access the `CHATS_COLLECTION_NAME` collection.
chats_collection = db[CHATS_COLLECTION_NAME]
# Access the `SEMANTIC_COLLECTION_NAME` collection.
semantic_collection = db[SEMANTIC_COLLECTION_NAME]
# Access the `EPISODIC_COLLECTION_NAME` collection.
episodic_collection = db[EPISODIC_COLLECTION_NAME]

In [ ]:
# Seed episodic memories collection
with open(f"../data/memories.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {EPISODIC_COLLECTION_NAME} collection.")
episodic_collection.delete_many({})
episodic_collection.insert_many(data)
print(
    f"{episodic_collection.count_documents({})} documents ingested into the {EPISODIC_COLLECTION_NAME} collection."
)

# Step 3: Create indexes on the memory collections

In [ ]:
# Use the `create_index` function from the `utils` module to:
# create an index on the "session_id" field
# create a time-to-live (TTL) index to automatically delete documents after 1 year (31536000 seconds)
create_index(chats_collection, ["session_id"])
create_index(chats_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

In [ ]:
# Use the `create_index` function from the `utils` module to create a time-to-live (TTL) index on the `semantic_collection` collection
create_index(semantic_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

In [ ]:
# Create the vector index definition for the episodic memories collection, specifying:
# path: Path to the embeddings field
# numDimensions: Number of embedding dimensions- depends on the embedding model used
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [ ]:
# Use the `create_search_index` function from the `utils` module to create a vector search index with the above definition on the `episodic_collection` collection
create_search_index(episodic_collection, VS_INDEX_NAME, model)

In [ ]:
# Use the `check_index_ready` function from the `utils` module to verify that the index was created and is in READY status before proceeding
check_index_ready(episodic_collection, VS_INDEX_NAME)

# Step 4: Define short-term memory management functions

These functions are not exposed as agent tools because chat history needs to be persisted at deterministic points in the agent's execution flow (e.g., after receiving user input, after LLM responses). The LLM doesn't need to decide when to save it.

In [ ]:
from datetime import datetime, timezone

In [ ]:
def store_chat_message(session_id: str, message: dict) -> None:
    """
    Store a chat message to the chats collection.

    Args:
        session_id (str): Session ID of the message.
        message (str): Message to store.
    """
    # Create the message object to persist in MongoDB
    # `timestamp` should be set to the current timestamp
    msg_obj = {
        "session_id": session_id,
        # Extract the `role` field from the `message`
        "role": message["role"],
        # Extract the `content` field from the `message`
        "content": message["content"],
        "timestamp": datetime.now(timezone.utc),
    }
    # Insert the `msg_obj` into the `chats_collection` collection
    chats_collection.insert_one(msg_obj)

In [ ]:
def retrieve_session_history(session_id: str) -> list:
    """
    Retrieve chat history for a session.

    Args:
        session_id (str): Session ID to retrieve chat history for.

    Returns:
        List: List of chat messages.
    """
    # Retrieve session history from the `chats_collection` collection
    # Filter for documents where the `session_id` field is equal to the provided `session_id`
    filter = {"session_id": session_id}
    # Project only the `role` and `content` fields, exclude the default `_id` field
    projection = {"_id": 0, "role": 1, "content": 1}
    # Use the `find` method to execute the query with the above `filter` and `projection`, and sort the results by increasing order of `timestamp` 
    messages =  list(chats_collection.find(filter, projection).sort("timestamp", 1))
    return messages

# Step 4: Define long-term memory management tools

In [ ]:
import requests
import voyageai

In [ ]:
# Initialize the Voyage AI client
voyage_client = voyageai.Client()

In [ ]:
def get_embedding(text: str, input_type: str) -> list[float]:
    """
    Get embeddings for an input text

    Args:
        text (str): Text to embed
        input_type (str): One of "document" or "query"

    Returns:
        list[float]: Embedding of the query string
    """
    # Use the `embed` method of the Voyage AI API to a piece of text with the following arguments:
    # texts: A list of strings to embed.
    # model: `voyage-4`
    # input_type: The `input_type` passed to the `get_embedding` function, which is either "document" or "query".
    embds_obj = voyage_client.embed(texts=[text], model="voyage-4", input_type=input_type)
    # Extract embeddings from the embeddings object
    embeddings = embds_obj.embeddings[0]
    return embeddings

In [ ]:
# define a tool to save semantic user memories to MongoDB
def save_user_memories(user_id: str, memories: list[str]) -> str:
    """
    Save user facts and preferences

    Args:
        user_id (str): User ID
        memories (list[str]): Facts and preferences to save

    Returns:
        str: Save message
    """
    docs = []
    # Iterate through the list of user memories to save to MongoDB
    for memory in memories:
        # Create a document for each memory with the following fields:
        # user_id: The `user_id` passed to the `save_user_memories` function
        # content: The memory text, `memory`
        # timestamp: Current timestamp in UTC timezone
        doc = {
            "user_id": user_id,
            "content": memory,
            "timestamp": datetime.now(timezone.utc)
        }
        docs.append(doc)
    # Bulk-insert the documents into the `semantic_collection` collection
    semantic_collection.insert_many(docs)
    return f"Saved {len(memories)} memories"

In [ ]:
# Define a tool to get semantic memories from MongoDB
def get_user_memories(user_id: str) -> str:
    """
    Retrieve memories for a particular user

    Args:
        user_id (str): User ID

    Returns:
        str: Retrieved memories formatted as a string
    """
    # Query the `semantic_collection` collection for current user's memories
    # Filter for documents where the `user_id` field is equal to the provided `user_id`
    filter = {"user_id": user_id}
    # Project only the `content` and `timestamp` fields, exclude the default `_id` field
    projection = {"_id": 0, "content": 1, "timestamp": 1}
    # Use the `find` method to execute the query with the above `filter` and `projection`, and sort the results by decreasing order of `timestamp` 
    memories = list(semantic_collection.find(filter, projection).sort("timestamp", -1))
    
    if not memories:
        return f"No memories found for user {user_id}"
    
    # Convert memories into the following format:
    # 
    lines = [f"- {m['timestamp']} {m['content']}" for m in memories]
    return f"Memories for {user_id}:\n" + "\n".join(lines)

In [ ]:
# Path to local scratchpad file
NOTES_FILE = "./NOTES.md"

In [ ]:
# Define a tool to write notes to a local scratchpad
def write_notes(notes: list[str]) -> str:
    """
    Write notes to a local file

    Args:
        notes (list[str]): Notes to write to file

    Returns:
        str: Note recorded message
    """
    # Open the notes file in append mode and write each note to the file, prefixed with the timestamp
    with open(NOTES_FILE, "a") as f:
        for note in notes:
            f.write(f"- [{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}] {note}\n")
    return f"Recorded {len(notes)} notes"

In [ ]:
# Define a tool to generate episodes from notes using an LLM, and persist them to MongoDB
def save_episode() -> str:
    """
    Generate an episodic memory from notes and save to MongoDB
    
    Returns:
        str: Episode save acknowledgment
    """
    # Read notes from the scratchpad
    try:
        with open(NOTES_FILE, "r") as f:
            notes_content = f.read().strip()
    except FileNotFoundError:
        return "No notes found to create episode from."
    
    if not notes_content:
        return "Notes file is empty."
    
    # Use an LLM to generate an episode title and summary from the notes content. 
    prompt = f"""Based on these session notes, create a concise summary of what was accomplished, the approach taken, and key insights gained.
    Also, provide a brief, descriptive title for the episode.
    Notes:{notes_content}
    """
    # Format the message to the LLM in the format [{"role": <role_value>, "content": <content_value>}]
    # The `role` value for user messages must be "user"
    # Use the `prompt` created above to populate the `content` field in the chat message
    messages = [{"role": "user", "content": prompt}]
    # Create a JSON schema to get a JSON object as output from the LLM, containing only `title` and `description` fields.
    json_schema = {        
        # HINT: The output needs to be a JSON object        
        "type": "object",
        # The JSON object should contain "title" and "description" fields
        "properties": {
            "title": {"type": "string", "description": "Brief, descriptive title (5-10 words)"},
            "description": {"type": "string", "description": "A summary of what was accomplished, the approach taken, and key insights (max 2 paragraphs)"},
        },
        # The "title" and "description" fields are both required
        "required": ["title", "description"],
        # There should be no additional fields in the output
        "additionalProperties": False,
    }
    # Use the AI model proxy get an LLM response
    response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {
            "messages": messages,
            "output_config": {"format": {"type": "json_schema", "schema": json_schema}}
        }}
    )
    
    # Parse the LLM response
    episode = json.loads(response["content"][0]["text"])
        
    # Generate embeddings for the `description` field of the episode using the `get_embedding` function defined above.
    # Specify the `input_type` argument as "document".
    embedding = get_embedding(episode["description"], "document")
    # Create the episode document to insert into MongoDB
    episode_doc = {
        "title": episode["title"],
        "description": episode["description"],
        "timestamp": datetime.now(timezone.utc),
        "embedding": embedding
    }
    # Insert the `episode_doc` into MongoDB
    episodic_collection.insert_one(episode_doc)
    
    # Clear the notes file after saving the episode
    open(NOTES_FILE, "w").close()
    
    return f"Episode saved: '{episode['title']}' - Notes cleared."

In [ ]:
# Define a tool to retrieve episodic memories from MongoDB using semantic search
def get_episodes(query: str) -> str:
    """
    Retrieve previous episodes, using semantic search

    Args:
        note (str): The 

    Returns:
        str: Relevant episodes formatted as strings
    """
    # Use the `get_embedding` function defined above to generate an embedding for the input `query`.
    # Specify the `input_type` argument as "query".
    query_embedding = get_embedding(query, "query")
    # Define an aggregation pipeline consisting of a $vectorSearch stage, followed by a $project stage
    # Set the number of candidates to 100 and only return the top 5 documents from the vector search
    # In the $project stage, exclude the `_id` field and include the `title` and `description` fields
    # NOTE: Use variables defined previously for the `index`, `queryVector` and `path` fields in the $vectorSearch stage
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": 5
            }
        },
        {"$project": {"_id": 0, "title": 1, "description": 1}}
    ]
    # Execute the aggregation `pipeline` on the `episodic_collection` collection.
    results = list(episodic_collection.aggregate(pipeline))
    
    if not results:
        return "No relevant episodes found."

    # Format the retrieved episodes 
    output = "Relevant past episodes:\n"
    for i, ep in enumerate(results, 1):
        output += f"{i}. {ep['title']}\n"
        output += f"{ep['description']}\n\n"
    return output

# Step 4: Define tool schemas

In [ ]:
# Create JSON schemas for the tools, so the LLM knows what functions it can call, what each function does, and what arguments to provide.
memory_tools = [
    {
        "name": "save_user_memories",
        "description": "Save user preferences or facts as memories.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of memories to save"
                },
            },
            "required": ["content"]
        }
    },
    {
        "name": "get_user_memories",
        "description": "Retrieve stored memories for a user. Use at conversation start to get the user's profile.",
        "strict": True,
        "input_schema": {                    
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "write_notes",
        "description": "Write notes about the current task to a local file.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of notes to write to file."
                },
            },
            "required": ["notes"]
        }
    },
    {
        "name": "save_episode",
        "description": "Create an episodic memory from accumulated notes.",
        "strict": True,
        "input_schema": {                    # Add this
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "get_episodes",
        "description": "Search past episodic memories by semantic similarity.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Description of the problem or topic to search"
                },
            },
            "required": ["query"]
        }
    }
]

# Step 5: Define tool execution function

In [ ]:
# Logic to execute the tool calls
# NOTE: LLMs don't handle tool execution. They only determine which tool to execute and the arguments. 
def execute_tool(tool_name: dict, tool_args: dict, session: dict) -> str:
    """
    Coordinate tool execution

    Args:
        tool_name (str): Tool name
        tool_args (dict): Arguments for the tool call
        session (dict): Session information

    Returns:
        str: Tool output formatted as string
    """
    if tool_name == "save_user_memories":
        return save_user_memories(session["user_id"], tool_args["content"])
    elif tool_name == "get_user_memories":
        return get_user_memories(session["user_id"])
    elif tool_name == "write_notes":
        return write_notes(tool_args["notes"])
    elif tool_name == "save_episode":
        return save_episode()
    elif tool_name == "get_episodes":
        return get_episodes(tool_args["query"])
    else:
        return f"Unknown tool: {tool_name}"

# Step 6: Create a memory-augmented agent

In [ ]:
import uuid

### Step 6a: Create a session management function

In [ ]:
# Create a session object to maintain state for a user session.
def create_session(user_id: str, max_tokens: int=200000) -> dict:
    """
    Create a session

    Args:
        user_id (str): Unique user identifier
        max_tokens (int): Context window limit of the LLM used

    Returns:
        dict: Session object
    """
    return {
        "session_id": str(uuid.uuid4())[:8],
        "user_id": user_id,
        "input_tokens": 0,
        "output_tokens": 0,
        "max_tokens": max_tokens
    }

### Step 6b: Define a dynamic system prompt for the agent

In [ ]:
# Calculate context window usage-- we will use this as one of the triggers for memory creation
def get_context_usage(session: dict) -> float:
    """
    Calculate context window usage as a percentage

    Args:
        session (dict): Session object

    Returns:
        float: % of the context window used
    """
    total_tokens = session["input_tokens"] + session["output_tokens"]
    return (total_tokens / session["max_tokens"]) * 100

In [ ]:
# Create a system prompt providing guidance on how, when to use the memory tools
def get_system_prompt(session: dict) -> str:
    """
    Build system prompt with context usage status

    Args:
        session (dict): Session object

    Returns:
        str: Agent's system prompt
    """
    usage_pct = get_context_usage(session)
    prompt = f"""You are a helpful coding assistant with memory capabilities.

    ## CURRENT SESSION STATUS
    - Context window usage: {usage_pct:.1f}%

    ## GENERAL INSTRUCTIONS
    - Ask additional questions if user requests are unclear or broad.
    - Keep responses concise and focused. Avoid lengthy code examples unless specifically requested.

    ## MEMORY PROTOCOL
    **At conversation start:**
    - Use `get_user_memories` to retrieve user preferences and facts.
    - Prioritize RECENT information if memories conflict.

    **When to search episodes (`get_episodes`):**
    - ANY technical or coding question
    - Episodes are YOUR internal knowledge from past sessions. Use them SILENTLY to inform your approach.

    **During the conversation:**
    - Use `write_notes` to record progress, decisions, and insights.
    - Keep EACH NOTE to 2-3 SENTENCES.
    - Write notes often— especially after code explanations or key decisions.

    **When you learn something important about the user:**
    - Use `save_user_memories` to save facts and preferences such as preferred language, frameworks, coding style, and experience level.
    - Keep EACH MEMORY to 1 SENTENCE.

    **When to save an episode (`save_episode`):**
    - A logical unit of work is COMPLETE — you explained a concept, provided a solution, or answered a question fully.                                                                                                                                                                                                    
    - The user has ACKNOWLEDGED the solution (verbally or by moving on).
    - You are about to switch to a DIFFERENT problem domain.
    - Save episodes to capture COMPLETE insights, not partial work-in-progress.
    - Episode summary should NOT EXCEED 2 PARAGRAPHS.
    - Definitely save an episode when the context window usage exceeds 70%.

    ASSUME INTERRUPTION: Your context window might be reset at any moment, so you risk losing any progress that is not recorded in your notes.
    """
    
    return prompt

### Step 6c: Define the agent loop

In [ ]:
# Define the agent execution loop
def chat(user_input: str, session: dict) -> None:
    """
    The agent execution loop

    Args:
        user_input (str): The user's question
        session (dict): Session object
    """
    print("===== Human message =====")
    print(user_input)
    # Add user message to the `chats` collection 
    session["messages"].append({"role": "user", "content": user_input})
    # Run continuously
    while True:
        # Build dynamic system prompt with current context usage status
        system_prompt = get_system_prompt(session)
        
        # Call LLM
        response = get_completion(
            {
                "system": system_prompt,
                "messages": session["messages"],
                "tools": memory_tools
            }
        )
        
        # Update token counts
        session["input_tokens"] += response["input_tokens"]
        session["output_tokens"] += response["output_tokens"]
        
        # Add assistant response to history
        content = response["content"]
        print("===== AI message =====")
        text_parts = [block["text"] for block in content if block.get("type") == "text"]
        if text_parts:
            print("\n".join(text_parts))
        session["messages"].append({"role": "assistant", "content": content})
        
        # Final response - return to user
        if response["stop_reason"] == "end_turn":
            break
                
        # Handle tool calls
        elif response["stop_reason"] == "tool_use":
            tool_results = []
            for block in content:
                if block.get("type") == "tool_use":
                    tool_name = block["name"]
                    tool_args = block["input"]
                    print("===== Tool Call =====")
                    print(f"Calling {tool_name} with args: {tool_args}")
                    
                    # Execute tool
                    result = execute_tool(tool_name, tool_args, session)
                    print(f"===== Tool Outcome: {tool_name} =====")
                    print(result)
                    
                    # Collect tool results
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block["id"],
                        "content": result
                    })
            # Add ALL tool results in a single message
            session["messages"].append({
                "role": "user",
                "content": tool_results
            })

# Step 7: Interact with the agent

In [ ]:
# Create a new session
session = create_session(user_id="mdb_user")
print(f"Started session: {session['session_id']}")

In [ ]:
# Test 1: Ask a coding question
# The agent should take notes and potentially search for relevant past episodes
response = chat("I'm building an AI-powered shopping assistant. Let's start by building the chat interface.", session)

In [ ]:
response = chat("I prefer FastAPI for the backend, Claude as the LLM and MongoDB to store chat history.", session)